# LangChain JSON Loader and Types

LangChain provides utilities for loading and parsing JSON data, which is essential for building data-driven applications and workflows. The JSON loader in LangChain helps you ingest structured data and convert it into usable document objects for further processing.

## Built-in JSON Loader

LangChain's built-in JSON loader can read JSON files and convert them into a list of `Document` objects. Each document typically contains the content and associated metadata.

**Example Usage:**
```python
from langchain.document_loaders import JSONLoader

loader = JSONLoader(file_path="data.json")
documents = loader.load()
```

## JSON Loader Types

- **Standard JSON Loader:** Loads each JSON object as a separate document.
- **Custom Schema Loader:** Allows you to specify how to extract content and metadata from each JSON object.

**Custom Schema Example:**
```python
from langchain.document_loaders import JSONLoader

def custom_content_func(json_obj):
    return json_obj["text"]

def custom_metadata_func(json_obj):
    return {"author": json_obj["author"], "date": json_obj["date"]}

loader = JSONLoader(
    file_path="data.json",
    content_key=custom_content_func,
    metadata_func=custom_metadata_func
)
documents = loader.load()
```

## Creating Custom JSON Loaders

You can create your own JSON loader by subclassing `BaseLoader` and implementing the `load` method. This allows for advanced parsing logic, filtering, or transformation.

**Custom Loader Example:**
```python
from langchain.document_loaders.base import BaseLoader
from langchain.schema import Document
import json

class MyCustomJSONLoader(BaseLoader):
    def __init__(self, file_path):
        self.file_path = file_path

    def load(self):
        with open(self.file_path, "r") as f:
            data = json.load(f)
        documents = []
        for item in data:
            content = item.get("body", "")
            metadata = {"id": item.get("id"), "tags": item.get("tags", [])}
            documents.append(Document(page_content=content, metadata=metadata))
        return documents

loader = MyCustomJSONLoader("custom_data.json")
documents = loader.load()
```

## Comparison and Use Cases

| Loader Type            | Flexibility | Use Case Example                                  |
|------------------------|-------------|---------------------------------------------------|
| Standard JSON Loader   | Low         | Simple flat JSON files, quick ingestion            |
| Custom Schema Loader   | Medium      | Extracting specific fields, mapping metadata       |
| Custom Loader (Class)  | High        | Complex/nested JSON, advanced filtering, enrichment|

**Use Cases:**
- **Standard Loader:** When your JSON file is a list of simple objects (e.g., blog posts, product listings).
- **Custom Schema Loader:** When you need to extract specific fields or transform the data (e.g., extracting only "text" and "author" from each object).
- **Custom Loader:** When working with deeply nested structures, combining multiple fields, or applying custom logic (e.g., employee records with nested address and skills).

For example, in this notebook, `MyCustomJSONLoader` is used to process nested employee data from multiple companies, extracting relevant fields and formatting them for downstream tasks.

## Summary

- LangChain provides flexible JSON loading utilities.
- You can use built-in loaders or define custom logic for content and metadata extraction.
- Custom loaders allow you to tailor the ingestion process to your specific data schema and workflow needs.

In [1]:
import os
import json
from typing import List, Callable, Optional, Dict, Any

os.makedirs("data/json_files", exist_ok=True)

In [3]:
# Create sample JSON files company data, employees info as nested data, Company name take some reputed company names
company_data = [
    {
        "id": 1,
        "name": "X Tech Solutions",
        "location": "New York",
        "employees": [
            {"id": 1, 
             "name": "Alice", 
             "position": "Engineer",
             "skills": ["Python", "Machine Learning"],
                "address": {
                    "street": "123 Main St",
                    "city": "New York",
                    "zip": "10001"
                }
            },
            {"id": 2, 
             "name": "Bob", 
             "position": "Manager",
             "skills": ["Project Management"],
             "address": {
                 "street": "456 Elm St",
                 "city": "New York",
                 "zip": "10001"
             }
            }
        ]
    }, {
        "id": 2,
        "name": "Innovatech",
        "location": "San Francisco",    
        "employees": [
            {"id": 3, 
             "name": "Charlie", 
             "position": "Designer",
             "skills": ["UI/UX", "Graphic Design"],
             "address": {
                 "street": "789 Pine St",
                 "city": "San Francisco",
                 "zip": "94107"
             }
            },
            {"id": 4, 
             "name": "Diana", 
             "position": "Developer",
             "skills": ["JavaScript", "React"],
             "address": {
                 "street": "101 Oak St",
                 "city": "San Francisco",
                 "zip": "94107"
             }
            }
        ]   
    },{
        "id": 3,
        "name": "Global Dynamics",
        "location": "Chicago",
        "employees": [
            {"id": 5, 
             "name": "Eve", 
             "position": "Analyst",
             "skills": ["Data Analysis", "SQL"],
             "address": {
                 "street": "202 Maple St",
                 "city": "Chicago",
                 "zip": "60601"
             }
            },
            {"id": 6, 
             "name": "Frank", 
             "position": "Consultant",
             "skills": ["Business Strategy"],
             "address": {
                 "street": "303 Birch St",
                 "city": "Chicago",
                 "zip": "60601"
             }
            }
        ]
    }
]

with open("data/json_files/company_data.json", "w") as f:
    json.dump(company_data, f, indent=4)


In [4]:
# Create sample JSON files with login events, include user id, page, event, amount 
# Note add amount only for purchase event
login_events = [ 
    {"user_id": 1, "page": "home", "event": "login"},
    {"user_id": 2, "page": "products", "event": "view"},
    {"user_id": 1, "page": "cart", "event": "add_to_cart"},
    {"user_id": 3, "page": "checkout", "event": "purchase", "amount": 99.99},
    {"user_id": 2, "page": "home", "event": "logout"},
    {"user_id": 4, "page": "home", "event": "login"},
    {"user_id": 4, "page": "products", "event": "view"},
    {"user_id": 4, "page": "cart", "event": "add_to_cart"},
    {"user_id": 4, "page": "checkout", "event": "purchase", "amount": 49.99},
    {"user_id": 4, "page": "home", "event": "logout"}
]

with open("data/json_files/login_events.json", "w") as f:
    json.dump(login_events, f, indent=4)


**Json Processing Strategies**

In [8]:
from langchain_community.document_loaders import JSONLoader
import json

employee_loader = JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema='.[].employees[]', # jq to extract each employee object from all companies
    text_content=False # Get full JSON object as metadata (not just text content)
)

employee_json_docs = employee_loader.load()
print(f"Loaded {len(employee_json_docs)} employee documents")
print(employee_json_docs[0])  # Print the first document to see structure
employee_json_docs


Loaded 6 employee documents
page_content='{"id": 1, "name": "Alice", "position": "Engineer", "skills": ["Python", "Machine Learning"], "address": {"street": "123 Main St", "city": "New York", "zip": "10001"}}' metadata={'source': 'D:\\krishna-codejournal\\kcj-langchain-rag-starter\\src\\langchain\\document_loaders\\data\\json_files\\company_data.json', 'seq_num': 1}


[Document(metadata={'source': 'D:\\krishna-codejournal\\kcj-langchain-rag-starter\\src\\langchain\\document_loaders\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "Alice", "position": "Engineer", "skills": ["Python", "Machine Learning"], "address": {"street": "123 Main St", "city": "New York", "zip": "10001"}}'),
 Document(metadata={'source': 'D:\\krishna-codejournal\\kcj-langchain-rag-starter\\src\\langchain\\document_loaders\\data\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Bob", "position": "Manager", "skills": ["Project Management"], "address": {"street": "456 Elm St", "city": "New York", "zip": "10001"}}'),
 Document(metadata={'source': 'D:\\krishna-codejournal\\kcj-langchain-rag-starter\\src\\langchain\\document_loaders\\data\\json_files\\company_data.json', 'seq_num': 3}, page_content='{"id": 3, "name": "Charlie", "position": "Designer", "skills": ["UI/UX", "Graphic Design"], "address": {"street": "789 P

In [13]:
# Custom JSON Loader 
from langchain.document_loaders.base import BaseLoader
from langchain.schema import Document
from typing import List, Callable, Optional, Dict, Any
import json
import os
class MyCustomJSONLoader(BaseLoader):
    def __init__(self, file_path: str):
        self.file_path = file_path
        print(f"Initialized MyCustomJSONLoader with file: {self.file_path}")

    def load(self):
        print(f"Loading data from {self.file_path}")
        with open(self.file_path, "r") as f:
            data = json.load(f)
        documents = []
        # data is a list of companies, each with an 'employees' list
        for company in data:
            for emp in company.get('employees', []):
                content = f"Name: {emp.get('name')}\nPosition: {emp.get('position')}\nSkills: {', '.join(emp.get('skills', []))}\n"
                address = emp.get("address", {})
                if address:
                    content += f"Address: {address.get('street')}, {address.get('city')}, {address.get('zip')}\n"
                metadata = {
                    'source': self.file_path,
                    'data_type': 'employee',
                    'employee_id': emp.get('id'),
                    'employee_name': emp.get('name'),
                    'position': emp.get('position')
                }
                documents.append(Document(page_content=content, metadata=metadata))
        return documents

In [15]:
loader = MyCustomJSONLoader("data/json_files/company_data.json")
custom_employee_docs = loader.load()
print(f"Loaded {len(custom_employee_docs)} custom employee documents") 

Initialized MyCustomJSONLoader with file: data/json_files/company_data.json
Loading data from data/json_files/company_data.json
Loaded 6 custom employee documents


In [16]:
def custom_json_loader(file_path: str) -> List[Document]:
    print(f"Loading data from {file_path}")
    with open(file_path, "r") as f:
        data = json.load(f)
    documents = []
    for company in data:
        for emp in company.get('employees', []):
            content = f"Name: {emp.get('name')}\nPosition: {emp.get('position')}\nSkills: {', '.join(emp.get('skills', []))}\n"
            address = emp.get("address", {})
            if address:
                content += f"Address: {address.get('street')}, {address.get('city')}, {address.get('zip')}\n"
            metadata = {
                'source': file_path,
                'data_type': 'employee',
                'employee_id': emp.get('id'),
                'employee_name': emp.get('name'),
                'position': emp.get('position')
            }
            documents.append(Document(page_content=content, metadata=metadata))
    return documents

In [21]:
custom_employee_docs = custom_json_loader("data/json_files/company_data.json")
print(f"Loaded {len(custom_employee_docs)} custom employee documents from function")
# for doc in custom_employee_docs:
#     print(doc.page_content)

print("Metadata for each document:")
for doc in custom_employee_docs:
    print(doc.metadata)

Loading data from data/json_files/company_data.json
Loaded 6 custom employee documents from function
Metadata for each document:
{'source': 'data/json_files/company_data.json', 'data_type': 'employee', 'employee_id': 1, 'employee_name': 'Alice', 'position': 'Engineer'}
{'source': 'data/json_files/company_data.json', 'data_type': 'employee', 'employee_id': 2, 'employee_name': 'Bob', 'position': 'Manager'}
{'source': 'data/json_files/company_data.json', 'data_type': 'employee', 'employee_id': 3, 'employee_name': 'Charlie', 'position': 'Designer'}
{'source': 'data/json_files/company_data.json', 'data_type': 'employee', 'employee_id': 4, 'employee_name': 'Diana', 'position': 'Developer'}
{'source': 'data/json_files/company_data.json', 'data_type': 'employee', 'employee_id': 5, 'employee_name': 'Eve', 'position': 'Analyst'}
{'source': 'data/json_files/company_data.json', 'data_type': 'employee', 'employee_id': 6, 'employee_name': 'Frank', 'position': 'Consultant'}
